# Training Data Generator

Generates synthetic datasets (case files, reviews, tickets, leads) via Hugging Face Inference API.

**Setup:** Add `HF_TOKEN` to `.env` and enable Inference Providers at https://hf.co/settings/inference-providers

In [21]:
%pip install -q huggingface_hub gradio python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.


In [22]:
import os
import json
import re
from datetime import datetime
from dotenv import load_dotenv
import gradio as gr
import pandas as pd
from huggingface_hub import InferenceClient

load_dotenv(".env", override=True)
HF_TOKEN = os.getenv("HF_TOKEN", "").strip()

In [23]:
MODELS = {
    "Llama 3 8B": "meta-llama/Meta-Llama-3-8B-Instruct",
    "Zephyr 7B": "HuggingFaceH4/zephyr-7b-beta",
    "Qwen 2.5 7B": "Qwen/Qwen2.5-7B-Instruct",
}

PROMPT_TEMPLATES = {
    "Investigation Cases": """Generate {N} fictional investigation case files for fraud/abuse/harassment. Return only a JSON array of {N} objects.
Fields: case_id (CASE-YYYY-####), created_at, region, intake_channel, case_type, short_summary, timeline, evidence, customer_messages, internal_notes, decision, rationale, followups, qa_tags.""",
    "Product Reviews": """Generate {N} fictional product reviews. Return only a JSON array of {N} objects.
Fields: product_id, product_name, category, rating (1-5), review_text, pros, cons, reviewer_region, date.""",
    "Support Tickets": """Generate {N} fictional support tickets. Return only a JSON array of {N} objects.
Fields: ticket_id, created_at, category, priority, subject, description, status, resolution, agent_notes.""",
    "Sales Leads": """Generate {N} fictional sales leads. Return only a JSON array of {N} objects.
Fields: lead_id, company_name, contact_name, email, phone, source, status, value, notes, created_at.""",
}

In [24]:
def extract_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```\w*\n?", "", text)
        text = re.sub(r"\n?```\s*$", "", text).strip()
    first, last = text.find("["), text.rfind("]")
    if first == -1:
        return None
    chunk = text[first : last + 1] if last > first else text[first:] + "]"
    try:
        return json.loads(chunk)
    except json.JSONDecodeError:
        return None

def generate(data_type, model_choice, num_records, temperature, max_tokens):
    if not HF_TOKEN:
        return "Error: HF_TOKEN missing. Add it to .env", None
    prompt = PROMPT_TEMPLATES.get(data_type, list(PROMPT_TEMPLATES.values())[0]).format(N=int(num_records))
    model_id = MODELS.get(model_choice, list(MODELS.values())[0])
    client = InferenceClient(model=model_id, token=HF_TOKEN)
    try:
        r = client.chat_completion(
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=temperature,
        )
        raw = (r.choices[0].message.content or "").strip()
    except Exception as e:
        return f"Error: {e}", None
    data = extract_json(raw)
    if data and isinstance(data, list):
        df = pd.DataFrame(data)
        return json.dumps(data, indent=2, default=str), df
    return raw, None

In [25]:
def save_json(json_str):
    if not json_str or json_str.startswith("Error"):
        return None
    fname = f"synthetic_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(fname, "w", encoding="utf-8") as f:
        f.write(json_str)
    return fname

def save_csv(df):
    if df is None:
        return None
    fname = f"synthetic_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    df.to_csv(fname, index=False)
    return fname

theme = gr.themes.Soft(primary_hue="green", secondary_hue="emerald")

with gr.Blocks(title="Training Data Generator") as demo:
    gr.Markdown("# Training Data Generator")

    with gr.Row():
        with gr.Column(scale=1):
            data_type = gr.Dropdown(
                list(PROMPT_TEMPLATES.keys()),
                value=list(PROMPT_TEMPLATES.keys())[0],
                label="Data Type",
            )
            model_dd = gr.Dropdown(list(MODELS.keys()), value=list(MODELS.keys())[0], label="Model")
        with gr.Column(scale=1):
            num_records = gr.Slider(1, 10, value=3, step=1, label="Number of records")
            with gr.Row():
                temp_slider = gr.Slider(0.1, 1.0, value=0.7, step=0.1, label="Temperature")
                max_tokens = gr.Slider(256, 4096, value=2048, step=128, label="Max tokens")

    btn = gr.Button("Generate", variant="primary", size="lg")
    data_state = gr.State(value=None)

    with gr.Row():
        out = gr.Textbox(label="JSON Output", lines=12)
        with gr.Column(scale=0, min_width=140):
            dl_json_btn = gr.Button("⬇ JSON", size="sm", variant="secondary")
            dl_csv_btn = gr.Button("⬇ CSV", size="sm", variant="secondary")
            json_file = gr.File(label="")
            csv_file = gr.File(label="")
    table_out = gr.Dataframe(label="Table Preview", wrap=True)

    def run_gen(dt, m, n, t, mx):
        j, df = generate(dt, m, n, t, mx)
        return j, df, df if df is not None and not df.empty else pd.DataFrame()

    btn.click(
        fn=run_gen,
        inputs=[data_type, model_dd, num_records, temp_slider, max_tokens],
        outputs=[out, data_state, table_out],
    )
    dl_json_btn.click(fn=save_json, inputs=out, outputs=json_file)
    dl_csv_btn.click(fn=save_csv, inputs=data_state, outputs=csv_file)

demo.launch(theme=theme, css=".main .container { max-width: 900px !important; }")

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
